In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import requests
import faiss
from typing import List
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from langchain.tools.tavily_search import TavilySearchResults
from langchain.agents import AgentExecutor, create_react_agent
from langchain.llms.base import LLM
from langchain import hub
from smolagents import CodeAgent, HfApiModel
import smolagents.tools as tools_module
import json

# =====================================
# 🔹 Global System Prompt (Shared by All Agents)
# =====================================
SYSTEM_PROMPT = (
    "Provide only the most relevant factual response in 3-4 sentences (max 350 characters). "
    "Do NOT include introductions, disclaimers, or statements about being an AI. "
    "Do NOT comment on how questions are structured. "
    "Do NOT include personal beliefs, opinions, or subjective statements. "
    "Do NOT start with 'I believe,' 'I think,' or 'Based on available information.' "
    "Do NOT mention 'the given text' or anything about how the question is formed. "
    "Simply state the factual answer."
)


import re

def clean_response(response: str) -> str:
    """Cleans AI-generated responses by removing disclaimers, meta-comments, and incomplete sentences."""
    
    # List of phrases to remove
    remove_phrases = [
        "As an AI,", "I'm not capable of", "I cannot provide personal opinions",
        "Based on available information", "I can provide a factual response",
        "System: The given input", "The given text", "Given the given text",
        "I believe", "I think", "It is possible that", "I've come to the conclusion that"
    ]
    
    # Remove listed phrases
    for phrase in remove_phrases:
        response = re.sub(rf"(^|\b){re.escape(phrase)}", "", response, flags=re.IGNORECASE).strip()

    # Remove incomplete sentences (fixes SmolAgents' issue)
    response = re.sub(r"^[,.\s]+", "", response)  # Remove leading commas, periods, or spaces

    # Ensure response starts with a capital letter
    if response and response[0].islower():
        response = response.capitalize()

    return response




# =====================================
# 🔹 PostgreSQL Database Configuration
# =====================================
DB_CONFIG = {
    "dbname": "agents_db",
    "user": "gauraangmalik",
    "password": "your_password",
    "host": "localhost",
    "port": 5432
}

# =====================================
# 🔹 Setup Database: Create Table (if it doesn't exist)
# =====================================
def setup_database():
    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS agents (
                    id SERIAL PRIMARY KEY,
                    title TEXT UNIQUE,
                    generated_response TEXT[]
                )
            """)
            conn.commit()

# =====================================
# 🔹 Store or Update Agent Responses in PostgreSQL
# =====================================
def store_agent_response(agent_name: str, response):
    if isinstance(response, dict):  
        response = str(response)

    with psycopg2.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cursor:
            cursor.execute("SELECT generated_response FROM agents WHERE title = %s", (agent_name,))
            result = cursor.fetchone()

            if result:
                current_responses = result[0] or []
                if response not in current_responses:
                    cursor.execute("""
                        UPDATE agents 
                        SET generated_response = generated_response || ARRAY[%s] 
                        WHERE title = %s
                    """, (response, agent_name))
                    print(f"✅ Response appended for agent: {agent_name} - {response}")
            else:
                cursor.execute("""
                    INSERT INTO agents (title, generated_response) 
                    VALUES (%s, ARRAY[%s])
                """, (agent_name, response))
                print(f"✅ New response stored for agent: {agent_name} - {response}")

            conn.commit()

# =====================================
# 🔹 Agent Functions (Refactored to Accept Model)
# =====================================

def run_chatcot(query_text: str, model: str) -> str:
    print(f"🤖 ChatCoT ({model}) is handling this query with Chain-of-Thought reasoning...")

    full_prompt = f"System: {SYSTEM_PROMPT}\nUser: {query_text}\nAssistant:"

    payload = {"model": model, "prompt": full_prompt, "stream": False}

    try:
        response = requests.post("http://localhost:11434/api/generate", json=payload)
        if response.status_code == 200:
            answer = response.json().get("response", "").strip()
            answer = clean_response(answer)  # ✅ Final cleaning

            if len(answer) > 350:
                answer = answer[:350] + "..."

            print(f"📝 ChatCoT Response: {answer}")
            return answer if answer else "No valid response received."

    except Exception as e:
        print(f"❌ ChatCoT request failed: {str(e)}")

    return "No valid response received."



def run_autogen(query_text: str, model: str) -> str:
    print(f"🤖 AutoGen ({model}) is handling this query...")

    full_prompt = f"System: {SYSTEM_PROMPT}\nUser: {query_text}\nAssistant:"

    payload = {"model": model, "prompt": full_prompt, "stream": False}

    try:
        response = requests.post("http://localhost:11434/api/generate", json=payload)
        if response.status_code == 200:
            answer = response.json().get("response", "").strip()
            answer = clean_response(answer)  # ✅ Final cleaning step

            if len(answer) > 350:
                answer = answer[:350] + "..."

            print(f"📝 AutoGen Response: {answer}")
            return answer if answer else "No valid response received."

    except Exception as e:
        print(f"❌ AutoGen request failed: {str(e)}")

    return "No valid response received."


import json  

def run_smolagents(query_text: str, model: str) -> str:
    print(f"🤖 SmolAgents ({model}) is handling this query...")

    full_prompt = f"System: {SYSTEM_PROMPT}\nUser: {query_text}\nAssistant:"

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": full_prompt},
            stream=True  
        )

        final_response = ""

        for line in response.iter_lines():
            if line:
                try:
                    parsed_line = json.loads(line.decode("utf-8"))
                    final_response += parsed_line.get("response", "")
                except json.JSONDecodeError as e:
                    print(f"❌ JSON Parsing Error: {e}")

        final_response = clean_response(final_response)  # ✅ Final cleaning

        if len(final_response) > 350:
            final_response = final_response[:350] + "..."

        print(f"📝 Final SmolAgents Response: {final_response}")
        return final_response if final_response else "No valid response received."

    except Exception as e:
        print(f"❌ SmolAgents Request Failed: {str(e)}")
        return "No valid response received."



def run_reflexion(query_text: str, model: str) -> str:
    print(f"🤖 Reflexion ({model}) is handling this query...")

    full_prompt = f"System: {SYSTEM_PROMPT}\nUser: {query_text}\nAssistant:"

    payload = {"model": model, "prompt": full_prompt, "stream": False}

    try:
        response = requests.post("http://localhost:11434/api/generate", json=payload)
        if response.status_code == 200:
            answer = response.json().get("response", "").strip()
            answer = clean_response(answer)  # ✅ Final cleaning

            if len(answer) > 350:
                answer = answer[:350] + "..."

            print(f"📝 Reflexion Response: {answer}")
            return answer if answer else "No valid response received."

    except Exception as e:
        print(f"❌ Reflexion request failed: {str(e)}")

    return "No valid response received."




# =====================================
# 🔹 Process Agents & Store Their Responses
# =====================================
def process_agents(query_text: str, model: str):
    """Ensures AutoGen, SmolAgents, Reflexion, and ChatCoT all run together and store responses once per agent."""
    setup_database()  # Ensure DB is ready

    agents = ["AutoGen", "SmolAgents", "Reflexion", "ChatCoT"]
    responses = {}

    for agent_name in agents:
        print(f"\n🔹 Processing {agent_name}...")
        if agent_name == "AutoGen":
            response = run_autogen(query_text, model)
        elif agent_name == "SmolAgents":
            response = run_smolagents(query_text, model)
        elif agent_name == "Reflexion":
            response = run_reflexion(query_text, model)
        elif agent_name == "ChatCoT":
            response = run_chatcot(query_text, model)

        responses[agent_name] = response if response else "No valid response received."

    # ✅ Clean responses once here (instead of inside each agent function)
    for agent_name, response in responses.items():
        responses[agent_name] = clean_response(response) if response else "No valid response received."

    # ✅ Store responses only if all agents returned valid results
    if all(responses.values()):
        for agent_name, response in responses.items():
            store_agent_response(agent_name, response)

        print("\n✅ Responses stored for all agents.")
    else:
        print("\n⚠️ Skipping storage: One or more agents failed to return a valid response.")


    # ✅ Print responses once per agent
    for agent_name, response in responses.items():
        print(f"\n✅ Response appended for agent: {agent_name} - {response}")

    # ✅ Store responses only if all agents returned valid results
    if all(responses.values()):
        for agent_name, response in responses.items():
            store_agent_response(agent_name, response)

        print("\n✅ Responses stored for all agents.")
    else:
        print("\n⚠️ Skipping storage: One or more agents failed to return a valid response.")


# =====================================
# 🔹 Main Function (Entry Point)
# =====================================
def main():
    model = "tinyllama"  # Change model name here when needed
    query_text = "Give me a few country names, but keep it short."
    process_agents(query_text, model)

if __name__ == "__main__":
    main()


When you scale up to multiple agents, some agents won't always return responses, and the system will later filter based on memory, planning, etc.

For now, everything is working as expected, but in the future, we’ll need to:

Handle missing responses gracefully (e.g., choosing fallback agents or weighting responses differently).
Ensure empty responses don’t break processing.
Optimize selection based on category matrix (memory, planning, etc.).

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

# =====================================
# 🔹 PostgreSQL Connection
# =====================================
DB_CONFIG = {
    "dbname": "agents_db",
    "user": "gauraangmalik",
    "password": "your_password",
    "host": "localhost",
    "port": 5432
}

# Initialize FAISS vector storage (Dictionary of FAISS Indexes)
faiss_indexes = {}

# Load SentenceTransformer Model for Vector Encoding
model = SentenceTransformer('all-MiniLM-L6-v2')

# =====================================
# 🔹 Step 1: Retrieve & Process Responses from PostgreSQL
# =====================================
conn = psycopg2.connect(**DB_CONFIG)
query = """
SELECT title, t.response, t.idx
FROM agents
CROSS JOIN LATERAL unnest(generated_response) WITH ORDINALITY AS t(response, idx)
WHERE title IN ('SmolAgents', 'AutoGen', 'Reflexion', 'ChatCoT')
ORDER BY idx;
"""

df = pd.read_sql(query, conn)
conn.close()

# =====================================
# 🔹 Step 2: Group Responses by Index
# =====================================
grouped_responses = defaultdict(dict)  # Dictionary to hold grouped responses

for _, row in df.iterrows():
    grouped_responses[row['idx']][row['title']] = row['response']  # Store responses per agent per index

# =====================================
# 🔹 Step 3: Convert Text Responses to Vectors & Store in FAISS
# =====================================
for idx, response_dict in grouped_responses.items():
    print(f"\n🔹 Processing Vector Space {idx}...")

    agent_names = list(response_dict.keys())
    responses = list(response_dict.values())

    if len(responses) < 2:
        print(f"⚠️ Skipping Vector Space {idx} - Not enough responses.")
        continue  # Skip vector spaces with fewer than 2 responses

    # Convert text responses to embeddings
    embeddings = model.encode(responses)

    # Initialize FAISS index if not already created
    dimension = embeddings.shape[1]
    if idx not in faiss_indexes:
        faiss_indexes[idx] = faiss.IndexFlatL2(dimension)

    # Add vectors only if they're not already stored
    if faiss_indexes[idx].ntotal == 0:
        faiss_indexes[idx].add(np.array(embeddings, dtype=np.float32))
        print(f"✅ Stored {len(responses)} responses in Vector Space {idx}")

# =====================================
# 🔹 Compute Similarity Metrics & Convergence Tracking
# =====================================
def check_stability(euclidean_distance, cos_sim, agent_pair, vector_space_id):
    """
    Determines if the agents are converging (stable) or producing very different results.
    """
    if euclidean_distance < 0.3 and cos_sim > 0.9:
        status = f"✅ {agent_pair} are converging (Stable)"
    elif euclidean_distance > 1.0 and cos_sim < 0.5:
        status = f"❌ {agent_pair} are producing very different results (Diverging)"
    else:
        status = f"🔄 {agent_pair} have some differences but are somewhat aligned (Partial Agreement)"
    
    print(f"🔹 Stability Status for {agent_pair} in Vector Space {vector_space_id}: {status}\n")


def compute_similarity(vector_space_id):
    """
    Computes both Euclidean Distance and Cosine Similarity between available agents in a given vector space.
    """
    if vector_space_id in faiss_indexes:
        index = faiss_indexes[vector_space_id]  # FAISS index for the Vector Space
        num_vectors = index.ntotal  # Number of stored vectors

        if num_vectors < 2:
            print(f"⚠️ Not enough responses in Vector Space {vector_space_id} to compute similarity.")
            return

        print(f"🔍 Computing similarity for Vector Space {vector_space_id}...")

        # ✅ Retrieve stored vectors
        stored_vectors = np.zeros((num_vectors, index.d), dtype=np.float32)
        for i in range(num_vectors):
            index.reconstruct(i, stored_vectors[i])  # Retrieve stored embeddings

        # ✅ Dynamically map agent names to vectors
        agent_names = list(grouped_responses[vector_space_id].keys())
        vector_map = {agent_names[i]: stored_vectors[i] for i in range(num_vectors)}

        # ✅ Compute pairwise similarity dynamically
        agent_pairs = [(a, b) for i, a in enumerate(agent_names) for b in agent_names[i+1:]]
        for agent_a, agent_b in agent_pairs:
            v1, v2 = vector_map[agent_a], vector_map[agent_b]
            euclidean_distance = np.linalg.norm(v1 - v2)
            cos_sim = cosine_similarity([v1], [v2])[0][0]

            print(f"✅ Euclidean Distance ({agent_a} ↔ {agent_b}): {euclidean_distance:.4f}")
            print(f"✅ Cosine Similarity ({agent_a} ↔ {agent_b}): {cos_sim:.4f}")
            check_stability(euclidean_distance, cos_sim, f"{agent_a} ↔ {agent_b}", vector_space_id)


# =====================================
# 🔹 Run Similarity Computation for Each Vector Space
# =====================================
for vector_space_id in grouped_responses.keys():
    compute_similarity(vector_space_id)

print("\n🚀 All vector spaces processed successfully!")


import psycopg2
import pandas as pd

# ===============================
# 🔹 Database Connection
# ===============================
DB_CONFIG = {
    "dbname": "agents_db",
    "user": "gauraangmalik",
    "password": "your_password",  # Replace with actual password
    "host": "localhost",
    "port": 5432
}

# ===============================
# 🔹 Fetch and Clean Data
# ===============================
def fetch_and_clean_responses():
    """
    Fetch responses from PostgreSQL and remove agents
    that do not have responses for any task.
    """
    
    with psycopg2.connect(**DB_CONFIG) as conn:
        query = """
        SELECT title, t.response, t.idx, profile, memory, planning, action,
               capability_acquisition, social_science, natural_science, engineering, benchmark
        FROM agents
        CROSS JOIN LATERAL unnest(generated_response) WITH ORDINALITY AS t(response, idx)
        ORDER BY t.idx;
        """
        df = pd.read_sql(query, conn)
    
    # ✅ Remove Agents with No Responses (Empty or NaN)
    df_filtered = df[df["response"].notna() & (df["response"].str.strip() != "")]
    
    print(f"\n✅ Original Data: {df.shape[0]} rows")
    print(f"✅ Cleaned Data: {df_filtered.shape[0]} rows (Removed {df.shape[0] - df_filtered.shape[0]} empty responses)\n")
    
    # ✅ Drop agents that are completely missing across all tasks
    agents_with_valid_responses = df_filtered["title"].unique()
    df_filtered = df_filtered[df_filtered["title"].isin(agents_with_valid_responses)]
    
    # ✅ Save Cleaned Data to CSV (Optional)
    df_filtered.to_csv("cleaned_agent_responses.csv", index=False)
    print("✅ Cleaned responses saved as 'cleaned_agent_responses.csv'.")
    
    return df_filtered

# Execute the cleaning function
cleaned_df = fetch_and_clean_responses()



In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# ===============================
# 🔹 1) Database & Model Setup
# ===============================
DB_CONFIG = {
    "dbname": "agents_db",
    "user": "gauraangmalik",
    "password": "your_password",  # Replace with actual password
    "host": "localhost",
    "port": 5432
}

# Load the SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# ===============================
# 🔹 2) Fetch Data from PostgreSQL
# ===============================
def fetch_task_responses():
    """Retrieve generated responses grouped by task index, including extra columns."""
    with psycopg2.connect(**DB_CONFIG) as conn:
        query = """
        SELECT title, t.response, t.idx, profile, memory, planning, action,
               capability_acquisition, social_science, natural_science, engineering, benchmark
        FROM agents
        CROSS JOIN LATERAL unnest(generated_response) WITH ORDINALITY AS t(response, idx)
        ORDER BY t.idx;
        """
        df = pd.read_sql(query, conn)
    return df

df = fetch_task_responses()
print("✅ Data Fetched Successfully!")
print(df.head())

# ===============================
# 🔹 3) Define Feature Encoding & Normalization
# ===============================
CATEGORY_KEYS = ["profile", "memory", "planning", "action", "capability_acquisition",
                 "social_science", "natural_science", "engineering", "benchmark"]
CATEGORY_WEIGHTS = {key: 1 / len(CATEGORY_KEYS) for key in CATEGORY_KEYS}  # Equal weights

# ===============================
# 🔹 4) Train PCA & Global Normalization
# ===============================
all_responses = df["response"].tolist()
all_embeddings = model.encode(all_responses)

# ✅ Apply **Global MinMax Scaling** BEFORE PCA
scaler_global = MinMaxScaler()
all_embeddings_scaled = scaler_global.fit_transform(all_embeddings)

# ✅ Train PCA globally
pca_n_components = min(2, all_embeddings_scaled.shape[1])  # Ensures valid PCA dimensions
pca = PCA(n_components=pca_n_components)
pca.fit(all_embeddings_scaled)

# ✅ Transform all embeddings
pca_embeddings = pca.transform(all_embeddings_scaled)

# ✅ Apply a **single** global MinMaxScaler for all PCA outputs
scaler_pca = MinMaxScaler()
pca_embeddings_scaled = scaler_pca.fit_transform(pca_embeddings)

# ===============================
# 🔹 5) Compute Rankings Per Task Using Fixed PCA
# ===============================
task_matrix = defaultdict(dict)
unique_tasks = df["idx"].unique()

for task_id in unique_tasks:
    task_data = df[df["idx"] == task_id]
    task_titles = task_data["title"].tolist()
    task_responses = task_data["response"].tolist()

    # ✅ Extract PCA embeddings **based on index** in all_responses
    response_indices = [all_responses.index(resp) for resp in task_responses]
    reduced_vectors = pca_embeddings_scaled[response_indices]

    print(f"\n🔹 Task {task_id}: PCA Output Shape = {reduced_vectors.shape}")  # ✅ Debugging print

    # ✅ Store category scores and reduced vectors
    for i, title in enumerate(task_titles):
        if i >= len(reduced_vectors):  # ✅ Prevent IndexError
            print(f"❌ ERROR: Index {i} out of bounds for PCA output in Task {task_id}")
            continue

        category_scores = {
            key: abs(CATEGORY_WEIGHTS[key] * reduced_vectors[i, 0]) for key in CATEGORY_KEYS[:5]
        }
        category_scores.update({
            key: abs(CATEGORY_WEIGHTS[key] * reduced_vectors[i, 1]) for key in CATEGORY_KEYS[5:]
        })

        # ✅ Compute **Total Score** properly
        category_scores["Total Score"] = sum(category_scores.values())

        # ✅ Ensure `"Reduced Vector"` is properly stored
        category_scores["Reduced Vector"] = reduced_vectors[i]

        task_matrix[task_id][title] = category_scores

# ✅ Remove Empty Responses Before Similarity Computation
for task_id in list(task_matrix.keys()):
    for agent in list(task_matrix[task_id].keys()):
        if np.all(task_matrix[task_id][agent]["Reduced Vector"] == 0):
            print(f"❌ Removing {agent} from Task {task_id} due to missing vectors")
            del task_matrix[task_id][agent]

# ===============================
# 🔹 Update Task Matrix CSV to Store Agent Names Properly
# ===============================

# Convert Task Matrix to DataFrame with Agents as a Named Column
task_matrix_df = pd.DataFrame.from_dict(
    {(task_id, agent): task_matrix[task_id][agent] for task_id in task_matrix.keys() for agent in task_matrix[task_id].keys()},
    orient='index'
).reset_index()

# Rename columns appropriately
task_matrix_df.rename(columns={"level_0": "Task", "level_1": "Agents"}, inplace=True)

# Save the updated DataFrame to CSV
csv_filename = "normalized_task_matrix_fixed.csv"
task_matrix_df.to_csv(csv_filename, index=False)

print(f"\n✅ Task Matrix CSV Updated: {csv_filename} (Agents column added)")


# ===============================
# 🔹 7) Compute Cosine Similarity & Heatmap
# ===============================
print("\n🔹 Computing Cosine Similarity Between Agents...\n")

valid_vectors = []
valid_agents = []

for task_id in sorted(task_matrix.keys()):
    for agent in sorted(task_matrix[task_id].keys()):
        vector = task_matrix[task_id][agent].get("Reduced Vector", None)
        
        if vector is not None and np.any(vector):  # ✅ Ensure vector is not empty
            valid_agents.append(f"{agent} (Task {task_id})")
            valid_vectors.append(vector)
            print(f"  ✅ {agent} (Task {task_id}): Vector Sum = {np.sum(vector):.4f}")
        else:
            print(f"  ❌ {agent} (Task {task_id}): No valid response")

if len(valid_agents) < 2:
    print("\n⚠️ Not enough valid responses to compute similarity. At least 2 are required.")
else:
    # Compute Cosine Similarity
    valid_vectors = np.array(valid_vectors)
    similarity_matrix = cosine_similarity(valid_vectors)

    # ✅ Adjust heatmap size dynamically
    num_agents = len(valid_agents)
    plt.figure(figsize=(min(15, num_agents // 5), min(12, num_agents // 5)))

    sns.heatmap(
        similarity_matrix,
        annot=False,
        cmap="coolwarm",
        xticklabels=valid_agents if num_agents < 40 else valid_agents[::2],
        yticklabels=valid_agents if num_agents < 40 else valid_agents[::2]
    )

    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.title("Cosine Similarity Between Agents (PCA-Reduced Features)")
    plt.show()

# ===============================
# 🔹 8) Display Task Rankings (Fixed)
# ===============================
print("\n🔹 Task Rankings (Sorted by Total Score):")

for task_id in sorted(task_matrix.keys()):
    print(f"\n🔹 Task {task_id} Rankings:")

    # ✅ Ensure rankings are sorted by **absolute** total score (no negatives)
    sorted_agents = sorted(task_matrix[task_id].items(), key=lambda x: abs(x[1]["Total Score"]), reverse=True)

    for rank, (agent, scores) in enumerate(sorted_agents, 1):
        print(f"  {rank}. {agent}: Total Score = {scores['Total Score']:.4f}")


In [ ]:
import pandas as pd

# Convert rankings into a structured DataFrame
ranking_data = []
for task_id in sorted(task_matrix.keys()):
    sorted_agents = sorted(task_matrix[task_id].items(), key=lambda x: abs(x[1]["Total Score"]), reverse=True)
    
    for rank, (agent, scores) in enumerate(sorted_agents, 1):
        ranking_data.append([task_id, rank, agent, scores["Total Score"]])

# Create DataFrame
ranking_df = pd.DataFrame(ranking_data, columns=["Task ID", "Rank", "Agent", "Total Score"])

# Display neatly formatted rankings
print("\n========================================")
print("📌 Task Rankings (Sorted by Total Score)")
print("========================================\n")
print(ranking_df.to_string(index=False))

csv_filename = "task_rankings.csv"
ranking_df.to_csv(csv_filename, index=False)
print(f"\n✅ Rankings saved to {csv_filename} for future analysis!")



In [ ]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ===============================
# 🔹 Prepare Data for K-Means Clustering
# ===============================
task_ids = df["idx"].unique()  # Task indices
agents = ["AutoGen", "SmolAgents", "Reflexion", "ChatCoT"]

# Create a matrix where rows are tasks and columns are agents' total scores
score_matrix = []

# Filter tasks that have all agents
for task_id in task_ids:
    if all(agent in task_matrix[task_id] for agent in agents):
        task_scores = [task_matrix[task_id][agent]["Total Score"] for agent in agents]
        score_matrix.append(task_scores)

# Convert the score matrix into a numpy array
score_matrix = np.array(score_matrix)

# Normalize the score matrix to standardize the features (important for clustering)
scaler = StandardScaler()
score_matrix_scaled = scaler.fit_transform(score_matrix)

# Perform K-Means clustering (with 5 clusters, can be changed)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(score_matrix_scaled)

# Assign clusters to each agent for each task
df_clusters = pd.DataFrame(score_matrix_scaled, columns=agents, index=task_ids[:len(score_matrix)])
df_clusters['Cluster'] = kmeans.labels_

# Display the clusters
print("\n🔹 K-Means Cluster Assignment for Each Task:")
print(df_clusters)

# ===============================
# 🔹 Plot K-Means Clustering Results
# ===============================
plt.figure(figsize=(10, 6))
for agent in agents:
    plt.scatter(df_clusters.index, df_clusters[agent], c=df_clusters['Cluster'], cmap='viridis', label=agent)

plt.xlabel('Task Index')
plt.ylabel('Normalized Total Score')
plt.title('🔹 K-Means Clustering: Agent Performance Across Tasks')
plt.legend()
plt.show()

# ===============================
# 🔹 Additional Analysis: Task Index Distribution
# ===============================
print("\n🔹 Unique Task Indices:")
print(df["idx"].unique())



In [ ]:
import pandas as pd

# Load the CSV file
file_path = "normalized_task_matrix_fixed.csv"
task_matrix_df = pd.read_csv(file_path, header=0)

# Display column names for debugging
print("Columns in DataFrame:", task_matrix_df.columns)

# Rename columns if necessary
if 'Agents' in task_matrix_df.columns:
    agent_column = 'Agents'
elif 'agents' in task_matrix_df.columns:
    agent_column = 'agents'
elif 'Unnamed: 0' in task_matrix_df.columns:  
    agent_column = 'Unnamed: 0'
    task_matrix_df.rename(columns={'Unnamed: 0': 'Agents'}, inplace=True)  # Rename first column

if 'Task' in task_matrix_df.columns:
    task_column = 'Task'
elif 'task' in task_matrix_df.columns:
    task_column = 'task'
else:
    task_matrix_df.reset_index(inplace=True)
    task_matrix_df.rename(columns={'index': 'Task'}, inplace=True)

# Verify column existence before pivoting
if agent_column not in task_matrix_df.columns or task_column not in task_matrix_df.columns:
    raise KeyError(f"Columns '{agent_column}' or '{task_column}' not found in DataFrame.")

# Pivot the table: Tasks as rows, Agents as columns
task_agent_table = task_matrix_df.pivot(index=task_column, columns=agent_column, values="Total Score")

# Save the structured table
output_file = "task_agent_matrix.csv"
task_agent_table.to_csv(output_file)

print(f"\n✅ Task-Agent Table saved as '{output_file}'.")

# Display the first few rows of the table for verification
print("\n🔹 Preview of Task-Agent Table:")
print(task_agent_table.head())




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Load CSV file
file_path = "task_agent_matrix.csv"  # Update with actual path
df = pd.read_csv(file_path)

# Drop rows with missing values
df.dropna(inplace=True)

# Extract agent response vectors (assuming numerical columns represent vectors)
vector_data = df.iloc[:, 1:].values  # Skipping non-vector columns if needed

# Compute Cosine Similarity and Euclidean Distance
cos_sim_matrix = cosine_similarity(vector_data)
euclidean_dist_matrix = euclidean_distances(vector_data)

# Plot Cosine Similarity Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(cos_sim_matrix, annot=False, cmap="coolwarm")
plt.title("Cosine Similarity Heatmap")
plt.xlabel("Agent Responses")
plt.ylabel("Agent Responses")
plt.show()

# Plot Euclidean Distance Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(euclidean_dist_matrix, annot=False, cmap="coolwarm")
plt.title("Euclidean Distance Heatmap")
plt.xlabel("Agent Responses")
plt.ylabel("Agent Responses")
plt.show()



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load the CSV file
file_path = "task_agent_matrix.csv"  # Update with actual path
df = pd.read_csv(file_path)

# Drop rows with missing values
df.dropna(inplace=True)

# Extract task names (first column)
task_names = df.iloc[:, 0].values  # Store task names

# Extract agent response vectors (excluding task names)
vector_data = df.iloc[:, 1:].values  # Shape: (num_tasks, num_agents)

# Step 1: Normalize the data using Min-Max scaling to scale it between 0 and 1
scaler = MinMaxScaler()
normalized_data = scaler.fit_transform(vector_data)  # Normalizes each feature to the range [0, 1]

# Step 2: Compute Euclidean Distance for each task after normalization
task_comparisons = []
num_tasks, num_agents = normalized_data.shape

# Generate column labels dynamically using real agent names
column_labels = [
    f"Agent_{i+1} and Agent_{j+1}" 
    for i in range(num_agents) for j in range(i + 1, num_agents)
]

# For each task, compute the pairwise Euclidean distances between agents
for task_idx in range(num_tasks):
    task_data = normalized_data[task_idx, :]  # Extract a single task row (all agents)
    task_distances = []

    # Compute pairwise Euclidean distances between agents for this task
    for i in range(num_agents):
        for j in range(i + 1, num_agents):  # Avoid redundant calculations
            distance = np.linalg.norm(task_data[i] - task_data[j])  # L2 norm
            task_distances.append(distance)

    # Store task name with computed distances
    task_comparisons.append([task_names[task_idx]] + task_distances)

# Step 3: Create a DataFrame with correct labels
comparison_df = pd.DataFrame(task_comparisons, columns=["Task Name"] + column_labels)

# Step 4: Save to CSV file
csv_file_path = "task_agent_euclidean_distances_normalized.csv"
comparison_df.to_csv(csv_file_path, index=False)

print(f"Normalized Task-wise Euclidean Distances saved to: {csv_file_path}")


In [ ]:
import pandas as pd

# Load the CSV file with Euclidean Distances
file_path = "task_agent_euclidean_distances.csv"
df = pd.read_csv(file_path)

# Define a function to categorize the Euclidean distance
def categorize_distance(distance):
    if distance <= 0.10:
        return "Optimal 🏆"
    elif distance <= 0.20:
        return "Suboptimal ⚖️"
    else:
        return "Not Optimal ❌"

# Initialize an empty dictionary to store the top 3 agent pairs for each task
top_3_pairs_per_task = {}

# Iterate through each task and find the top 3 agent pairs with the shortest distance
for index, row in df.iterrows():
    task_name = row['Task Name']
    
    # Create a list to store agent pairs and their corresponding distances for the current task
    task_distances = []
    
    # Iterate over the agent comparison columns
    for col in df.columns[1:]:  # Skip the 'Task Name' column
        agent_pair = col.replace("vs", "and")  # Replace "vs" with "and"
        distance = row[col]
        task_distances.append((agent_pair, distance))
    
    # Sort the distances in ascending order (smallest distance first)
    sorted_task_distances = sorted(task_distances, key=lambda x: x[1])
    
    # Get the top 3 agent pairs with the shortest distance
    top_3_pairs_per_task[task_name] = sorted_task_distances[:3]

# Print the top 3 agent pairs for each task with their categories and emojis
for task, pairs in top_3_pairs_per_task.items():
    print(f"Task: {task}")
    for pair, dist in pairs:
        category = categorize_distance(dist)
        print(f"  {pair}: {dist} - {category}")
    print()  # New line for readability



we have Total Score, Reduced Vector 
we also have clustering agents based on tasks


In [ ]:
import requests  # For making HTTP requests
import re

# Define the system prompt
system_prompt = """
Given a task description, classify the task into two categories:
1. **Complexity**: (low, medium, high)
2. **Subjectivity**: (factual, subjective)

Please provide the task attributes as a one-word answer for each category, separated by a comma.
Example: Task description: "What is the capital of France?" => Complexity: low, Subjectivity: factual.
"""

# Define the API endpoint for TinyLlama (adjust this URL if needed)
api_url = "http://localhost:11434/api/generate"

# Prepare the full prompt to send to TinyLlama
full_prompt = system_prompt + "\nTask description: " + query_text + "\n"

# Create payload for the API request
payload = {"model": "tinyllama", "prompt": full_prompt, "stream": False}

# Send the POST request to the TinyLlama API
try:
    response = requests.post(api_url, json=payload)

    if response.status_code == 200:
        # Extract the response text from the API
        task_attributes = response.json().get("response", "").strip()

        # Print the raw response to help debug
        print(f"Raw response from TinyLlama: {task_attributes}")

        # Try to extract complexity and subjectivity from the response
        try:
            # Attempt to split based on the expected format
            parts = task_attributes.split(",")
            if len(parts) == 2:
                complexity = parts[0].split(":")[1].strip()
                subjectivity = parts[1].split(":")[1].strip()

                # Print the task attributes
                print(f"Task Attributes for '{query_text}':")
                print(f"Complexity: {complexity}")
                print(f"Subjectivity: {subjectivity}")

            else:
                # If we can't split it correctly, print a warning
                print("⚠️ Unable to extract complexity and subjectivity in expected format.")
                complexity = "unknown"
                subjectivity = "unknown"

        except Exception as e:
            print(f"❌ Error extracting task attributes: {str(e)}")
            complexity = "unknown"
            subjectivity = "unknown"

        # Function to categorize the Euclidean distance based on task attributes
        def categorize_distance(distance, complexity, subjectivity):
            if subjectivity == 'factual':
                if distance <= 0.10:
                    return "Optimal 🏆"
                elif distance <= 0.20:
                    return "Suboptimal ⚖️"
                else:
                    return "Not Optimal ❌"

            elif subjectivity == 'subjective':
                if distance <= 0.20:
                    return "Optimal 🏆"
                elif distance <= 0.40:
                    return "Suboptimal ⚖️"
                else:
                    return "Not Optimal ❌"

            if complexity == 'high':
                if distance <= 0.30:
                    return "Optimal 🏆"
                elif distance <= 0.50:
                    return "Suboptimal ⚖️"
                else:
                    return "Not Optimal ❌"

            # Default for medium complexity or unknown task types
            if distance <= 0.15:
                return "Optimal 🏆"
            elif distance <= 0.30:
                return "Suboptimal ⚖️"
            else:
                return "Not Optimal ❌"
            

        # Example Euclidean distance for task
        distance_example = 0.18
        category = categorize_distance(distance_example, complexity, subjectivity)
        print(f"Task Distance: {distance_example} - {category}")

    else:
        print(f"❌ Error: Unable to fetch response from TinyLlama (status code: {response.status_code})")

except Exception as e:
    print(f"❌ Exception occurred: {str(e)}")


Or i can ask the gen AI to make the type of tasks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file with Euclidean distances
file_path = 'task_agent_euclidean_distances.csv'  # Adjust path if necessary
df = pd.read_csv(file_path)

# Assuming 'Task Name' is the index, and other columns contain Euclidean distances between agent pairs
# Let's inspect the structure of the dataframe
print(df.head())

# Extract the columns and agent pairs
agents = df.columns[1:]  # Assuming first column is 'Task Name' and others are agent comparisons
tasks = df['Task Name']

# Create a figure and axis
plt.figure(figsize=(12, 8))

# Iterate through each agent pair and plot the Euclidean distances
for agent in agents:
    plt.scatter(tasks, df[agent], label=agent, alpha=0.7)

# Add labels and title
plt.xlabel('Task Index')
plt.ylabel('Euclidean Distance')
plt.title('🔹 Euclidean Distance Between Agent Responses Across Tasks')

# Show the legend to differentiate agents
plt.legend(title="Agents")

# Show the plot
plt.show()


In [ ]:
#Switching to SQL
import psycopg2
import pandas as pd

# Update these connection parameters with your actual details
conn = psycopg2.connect(
    host="localhost",
    database="agents_db",
    user="gauraangmalik",
    password="your_password"  # Replace with your PostgreSQL password
)

# Run a SQL query to fetch all data from the agents table
query = "SELECT * FROM agents;"
df = pd.read_sql(query, conn)

# Close the database connection
conn.close()

# Display the DataFrame
df


In [ ]:
import psycopg2

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="agents_db",
    user="gauraangmalik",
    password="your_password"  # Replace with your actual PostgreSQL password
)

# Clear the generated_response column (reset to empty array)
with conn:
    with conn.cursor() as cursor:
        cursor.execute("UPDATE agents SET generated_response = '{}';")
        conn.commit()

print("✅ All responses in generated_response have been cleared.")